# Emotion Machine — Memory Feature Validation

This notebook validates the end-to-end memory behavior for companions:
- Companion-level memory toggle persistence
- Core memories injection into the effective system prompt (but not the builder editor)
- Regular memory creation from messages (message_id-backed) and per-user retrieval
- Heuristic-driven memory retrieval (best-effort check when an API key is present)

It targets a running local API at `http://localhost:8100` by default. Set environment variables below to adjust.

In [ ]:
import json
import os

import requests
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.getenv("EM_API_BASE", "http://localhost:8100")
AUTH_BYPASS = os.getenv("DISABLE_AUTH_FOR_TESTING", "true").lower() == "true"
TOKEN = os.getenv("EM_TEST_TOKEN")  # if set, used as Bearer token


def api(path: str) -> str:
    return f"{BASE_URL}{path}"


def get_dev_token() -> str | None:
    try:
        r = requests.get(api("/api/auth/dev-token"), timeout=10)
        if r.ok:
            return r.json().get("token")
    except Exception as e:
        print("Dev token fetch failed:", e)
    return None


def headers(token: str | None = None):
    h = {"Content-Type": "application/json"}
    t = token or TOKEN
    if t:
        h["Authorization"] = f"Bearer {t}"
    return h


def pretty(o):
    print(json.dumps(o, indent=2, ensure_ascii=False))


print("BASE_URL =", BASE_URL)
print("AUTH_BYPASS =", AUTH_BYPASS)
print("OPENAI_API_KEY present =", bool(os.getenv("OPENAI_API_KEY")))

if AUTH_BYPASS and not TOKEN:
    TOKEN = get_dev_token()
    print("Using dev token:", bool(TOKEN))
else:
    print("Using provided token:", bool(TOKEN))

## 1) Ensure a test companion exists and enable memory

In [ ]:
# Find or create a companion named 'Memory Test Companion'
r = requests.get(api("/api/companions"), headers=headers(TOKEN), timeout=20)
r.raise_for_status()
comps = r.json()
test_comp = next((c for c in comps if c["name"] == "Memory Test Companion"), None)
if not test_comp:
    payload = {
        "name": "Memory Test Companion",
        "description": "Validation of memory features",
        "config": {
            "system_prompt": {
                "full_system_prompt": "You are a concise assistant. Keep answers short."
            },
            "voice": {
                "popular_options": ["OpenAI - speech-to-speech"],
                "voice": ["alloy"],
                "temperature": 0.25,
            },
            "memory": {
                "enabled": False,
                "core_memories": [],
                "memory_evaluation_prompt": "",
                "recency": 0.995,
                "top_k": 50,
                "min_saliency": 0.2,
            },
        },
    }
    cr = requests.post(
        api("/api/companions"), headers=headers(TOKEN), data=json.dumps(payload), timeout=30
    )
    cr.raise_for_status()
    cfg = cr.json()
    # Fetch list again to get id
    comps = requests.get(api("/api/companions"), headers=headers(TOKEN), timeout=20).json()
    test_comp = next((c for c in comps if c["name"] == "Memory Test Companion"), None)
assert test_comp, "Failed to create or locate test companion"
companion_id = test_comp["id"]
print("Companion ID:", companion_id)

# Enable memory + set guidance
cfg = requests.get(
    api(f"/api/companions/{companion_id}"), headers=headers(TOKEN), timeout=20
).json()
cfg["memory"]["enabled"] = True
cfg["memory"]["memory_evaluation_prompt"] = (
    "Remember stable user preferences and goals; ignore small talk."
)
ur = requests.put(
    api(f"/api/companions/{companion_id}"),
    headers=headers(TOKEN),
    data=json.dumps({"config": cfg}),
    timeout=40,
)
ur.raise_for_status()
print("Memory enabled and guidance saved.")

## 2) Seed core memories and verify effective system prompt

Note: We re-save the companion config after adding a core memory to snapshot a new DEPLOYED version whose effective_system_prompt includes the latest core memories.


In [ ]:
# Add a core memory
core_text = 'Always greet the user with their nickname "Ace".'
body = {"content": core_text, "sender_type": "system", "importance": 1.0, "is_core": True}
mr = requests.post(
    api(f"/api/companions/{companion_id}/memories"),
    headers=headers(TOKEN),
    data=json.dumps(body),
    timeout=20,
)
mr.raise_for_status()
print("Core memory added:", mr.json())

# Create a conversation to snapshot a prompt; this uses direct companion conversation
cr = requests.post(
    api("/conversations/"),
    headers=headers(TOKEN),
    data=json.dumps({"companion_id": companion_id}),
    timeout=20,
)
cr.raise_for_status()
conv = cr.json()
conversation_id = conv["id"]
external_user_id = conv.get("external_user_id")
print("Conversation created:", conversation_id, "external_user_id:", external_user_id)

# Read the effective system prompt from analytics endpoint (prefers effective_system_prompt)
spr = requests.get(
    api(f"/api/analytics/conversations/{conversation_id}/system-prompt"),
    headers=headers(TOKEN),
    timeout=20,
)
spr.raise_for_status()
sp = spr.json()["system_prompt"]
print("Effective system prompt length:", len(sp))
assert "# CORE MEMORIES" in sp and "Ace" in sp, "Core memories not found in effective system prompt"
print("✓ Effective system prompt contains core memories.")

# Confirm builder JSON is clean (no core memories appended)
cfg2 = requests.get(
    api(f"/api/companions/{companion_id}"), headers=headers(TOKEN), timeout=20
).json()
assert "# CORE MEMORIES" not in (cfg2["system_prompt"]["full_system_prompt"] or ""), (
    "Builder prompt leaked core memories"
)
print("✓ Builder prompt did not include core memories.")

In [ ]:
# Bump version so effective_system_prompt captures newly added core memories
cfg3 = requests.get(
    api(f"/api/companions/{companion_id}"), headers=headers(TOKEN), timeout=20
).json()
# No-op change: re-PUT the same config; server will create a new DEPLOYED version
requests.put(
    api(f"/api/companions/{companion_id}"),
    headers=headers(TOKEN),
    data=json.dumps({"config": cfg3}),
    timeout=40,
).raise_for_status()
print("Recreated DEPLOYED version to capture updated core memories.")

## 3) Create regular user memory from a message (message_id-backed)

In [ ]:
# Send a user message that should be worth remembering
user_fact = "My favorite color is green."
sr = requests.post(
    api(f"/conversations/{conversation_id}/messages"),
    headers=headers(TOKEN),
    data=json.dumps(
        {
            "content": user_fact,
            "system_prompt": cfg2["system_prompt"]["full_system_prompt"],
            "llm_provider": "openai-gpt4o-mini",
            "temperature": 0.2,
        }
    ),
    timeout=60,
)
sr.raise_for_status()
resp = sr.json()
print("User+Assistant messages created.")
pretty(resp)

# Verify a memory row was created (coalesced content returned)
mems = requests.get(
    api(f"/api/companions/{companion_id}/memories?limit=50&order_by=created_at&order_dir=DESC"),
    headers=headers(TOKEN),
    timeout=20,
).json()
assert any(("favorite color" in (m["content"] or "").lower()) for m in mems), (
    "Expected user fact memory not found"
)
target_mem = next(m for m in mems if "favorite color" in (m["content"] or "").lower())
print("✓ Regular memory present (message-backed). is_core=", target_mem["is_core"])

## 4) Per-user retrieval: search by query and external_user_id

In [ ]:
qr = (
    requests.post(
        api(f"/api/companions/{companion_id}/memories/search"),
        headers=headers(TOKEN),
        data=json.dumps(
            {
                "query": "favorite color",
                "external_user_id": external_user_id,
                "top_k": 10,
                "min_saliency": 0.2,
            }
        ),
    ),
)
qr = qr[0] if isinstance(qr, tuple) else qr
qr.raise_for_status()
items = qr.json()["items"]
print("Per-user search returned", len(items), "items")
if not items:
    print("No items via external_user_id; debugging external_user_ids from recent memories…")
    mems_dbg = requests.get(
        api(f"/api/companions/{companion_id}/memories?limit=20&order_by=created_at&order_dir=DESC"),
        headers=headers(TOKEN),
        timeout=20,
    ).json()
    ext_ids = sorted(set((m.get("external_user_id") or "") for m in mems_dbg))
    print("Recent memory external_user_ids:", ext_ids)
    print("Conversation external_user_id:", external_user_id)
    # Fallback: try conversation_id filter
    qr2 = requests.post(
        api(f"/api/companions/{companion_id}/memories/search"),
        headers=headers(TOKEN),
        data=json.dumps(
            {
                "query": "favorite color",
                "conversation_id": conversation_id,
                "top_k": 10,
                "min_saliency": 0.2,
            }
        ),
    )
    qr2.raise_for_status()
    items = qr2.json()["items"]
    print("Fallback by conversation_id returned", len(items), "items")
assert len(items) >= 1 and any("favorite color" in (i["content"] or "").lower() for i in items), (
    "Per-user search did not return expected memory"
)
print(f"✓ Per-user search returned {len(items)} items; top score=", items[0].get("score"))

## 5) Heuristic retrieval behavior (best-effort)
- If `OPENAI_API_KEY` is set, we expect the assistant to leverage the memory when asked directly.
- If not set, we skip response validation and only validate that search works and core memories are present in effective prompt.

In [ ]:
if os.getenv("OPENAI_API_KEY"):
    ask = "What is my favorite color?"
    sr2 = requests.post(
        api(f"/conversations/{conversation_id}/messages"),
        headers=headers(TOKEN),
        data=json.dumps(
            {
                "content": ask,
                "system_prompt": cfg2["system_prompt"]["full_system_prompt"],
                "llm_provider": "openai-gpt4o-mini",
                "temperature": 0.2,
            }
        ),
        timeout=60,
    )
    sr2.raise_for_status()
    a = sr2.json()["assistant_message"]["content"]
    print("Assistant reply:", a)
    assert "green" in a.lower(), (
        "Assistant did not appear to use retrieved memory (expected to mention green)"
    )
    print("✓ Assistant reply reflects retrieved memory (heuristic likely applied).")
else:
    print(
        "OPENAI_API_KEY not set — skipping heuristic response validation. Search and prompt checks already passed."
    )